# Production-Grade Prompting, Agents & Tool Use

Welcome back! This is the Module 2 lab, the follow-up to *MSO Foundations*.

Writing code that uses Claude is not the same as using Claude to write code. A prototype can work fine on your laptop and still break once real users, real documents and real traffic show up. In this lab we start with a small contract-review prototype and fix what breaks, one problem at a time.

Each chapter takes one problem you'll run into in production and shows how to fix it:

| # | Chapter | Production failure it prevents |
|---|---|---|
| 1 | Setup | — |
| 2 | Prompting craft | Output in the wrong shape breaks the parser |
| 3 | Extended thinking | Paying for reasoning you don't need |
| 4 | Tool use & schema design | Claude picks the wrong tool; the loop breaks |
| 5 | Streaming | A dropped stream corrupts history |
| 6 | Context engineering | The context window fills at turn 8 |
| 7 | Agent construction | An agent makes an irreversible change unchecked |
| 8 | Agent memory | State is lost (or bloats every call) between sessions |
| 9 | Multimodal & batch | Image costs and rate limits blow up at scale |
| 10 | Debug challenge | All of the above in one buggy agent |

Let's get started 🚀

### Prerequisite
Download [AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing) before you start. It's the short AWS contract from the MSO lab, and you'll upload it at the end of Chapter 1.

## Chapter 1 — Setup

First, the setup. We need to install four packages:

1. `anthropic` is the official Python library for talking to Claude.
2. `PyMuPDF` (called `fitz` in code) reads the text out of our PDF.
3. `pyboxen` prints output in neat boxes so it's easy to find as you scroll.
4. `pillow` lets us make test images in Chapter 9.

#### What are we going to do?

Run one install command. It takes a few seconds, and `-q` just keeps the output short.

In [ ]:
# Install the libraries: Claude SDK, PDF reader, pretty boxes, image tools (-q = quiet, -U = latest)
!pip install -q -U anthropic PyMuPDF pyboxen pillow

Next come the imports and a few small helpers that we'll use in every chapter.

#### What are we going to do?

1. Import the libraries we just installed.
2. Ask for your Anthropic API key. We use `getpass` so what you type stays hidden and doesn't get saved in the notebook.
3. Create `client`, the object we use to call Claude, and give names to the two models: Haiku (fast and cheap) and Sonnet (smarter).
4. Add two small helpers. `show()` prints text in a titled, colored box, and `text_of()` pulls the plain text out of Claude's reply.

When you run the cell, a box will ask for your key. Paste it in and press Enter.

In [ ]:
# Load the libraries we need
import os, json, math, base64, asyncio, time
from getpass import getpass

import anthropic
import fitz  # PyMuPDF
from pyboxen import boxen

# Ask for your API key (hidden) and store it where the SDK looks for it
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
# 'client' is our connection to Claude
client = anthropic.Anthropic()

# Nicknames for the two models: Haiku = fast & cheap, Sonnet = smarter
HAIKU, SONNET = "claude-haiku-4-5", "claude-sonnet-4-6"


# Helper: print text inside a titled, colored box
def show(title, text, color="cyan"):
    print(boxen(str(text), title=title, color=color, padding=1))


# Helper: Claude's reply is a list of blocks; keep only the text ones
def text_of(response):
    """Join all text blocks of a response."""
    return "".join(b.text for b in response.content if b.type == "text")

Now we load the contract. We'll use it in almost every chapter, so we only want to do this once.

#### What are we going to do?

1. Open the file picker so you can upload `AWS1.pdf`.
2. Read the PDF page by page and keep each page's text in a list called `pages`. Later our tools will need to grab single pages.
3. Join all the pages into one long string, `contract_text`, for when we want to give Claude the whole contract.
4. Print a quick check showing how many pages and characters we have.

In [ ]:
# Open the upload dialog and choose AWS1.pdf
from google.colab import files
files.upload()

# Read the PDF: one string of text per page
with fitz.open("/content/AWS1.pdf") as doc:
    pages = [p.get_text() for p in doc]
# Join all pages into one long string
contract_text = "\n".join(pages)
# Sanity check: how big is the contract?
print(f"{len(pages)} pages · {len(contract_text):,} characters")

## Chapter 2 — Prompting Craft

You've probably seen this: a prompt works great in the chat window, then breaks in production on an input you never tried. The first instinct is to add more words to the prompt. That usually doesn't help. A better move is to figure out what is actually missing:

| What you observe | What's missing | Fix |
|---|---|---|
| Wrong *shape* (a sentence instead of a label) | **Output constraint** | State the exact form, the allowed values, and "nothing else" |
| Scope or tone drifts across turns | **System prompt** | A persistent role plus rules |
| Right task, but Claude invents its own structure | **Few-shot examples** | Show one correct input → output pair |
| Instructions blend into the data | **XML tags** | Use `<clause>…</clause>` boundaries |

**Our task:** say we have a program that needs exactly one of four labels for each contract clause. If Claude answers with a long sentence instead, the program breaks. So instead of just reading the outputs and guessing, we'll count how often the output is usable. Usable means one valid label and nothing else.

#### What are we going to do?

1. Set up the four allowed labels and five sample clauses. One of the clauses is a bit tricky on purpose.
2. Write a helper called `valid_rate()` that classifies every clause three times and tells us what share of the answers were valid.
3. Run it with a bare prompt ("Classify the clause") to get a starting score. It will probably be bad, and that's fine. It gives us something to beat.

In [ ]:
# The only answers we accept
LABELS = {"TERMINATION", "PAYMENT", "LIABILITY", "OTHER"}

# Five practice clauses to classify
clauses = [
    "Either party may terminate this Agreement for convenience on 30 days' written notice.",
    "Customer shall pay all fees within 30 days of the invoice date.",
    "Neither party's aggregate liability shall exceed the fees paid in the prior 12 months.",
    "This Agreement is governed by the laws of the State of Washington.",
    "We may suspend your access if you breach this Agreement, and fees continue to accrue during suspension.",
]


# Test machine: classify every clause `runs` times, return the share of valid answers
def valid_rate(system, wrap, runs=3):
    """Classify every clause `runs` times; return (share of valid outputs, sample outputs)."""
    # Collect every reply here
    outs = []
    for clause in clauses:
        for _ in range(runs):
            # Ask Claude to classify one clause
            r = client.messages.create(
                model=HAIKU, max_tokens=60, system=system,
                messages=[{"role": "user", "content": wrap(clause)}],
            )
            outs.append(text_of(r).strip())
    # Valid = the reply is exactly one of the labels
    ok = sum(o in LABELS for o in outs)
    return ok / len(outs), outs


# Run it with a bare prompt (no rules, no examples) to get our baseline
bare_rate, bare_outs = valid_rate("You are a clause classifier. Classify the clause.", lambda c: c)
show("❌ Bare prompt", f"Valid outputs: {bare_rate:.0%}\n\nSamples:\n" + "\n".join("• " + o[:80] for o in bare_outs[:4]), "red")

Now let's fix it. We won't add more words. We'll add structure instead.

#### What are we going to do?

We'll rewrite the prompt with three changes and run the exact same test again:

1. A system prompt that sets the role and the rules.
2. A clear output rule: "Return only the label. No other text."
3. A few examples wrapped in XML tags, including the tricky "suspend service" case, so Claude can see how to handle it.

Then compare the new score with the first one.

In [ ]:
# Standing instructions: role + rules + worked examples
SYSTEM = """You are a contract clause classifier.
Classify each clause into exactly one of: TERMINATION, PAYMENT, LIABILITY, OTHER.
Return only the label. No other text.

<sample_input>Either party may end this agreement with 60 days' notice.</sample_input>
<ideal_output>TERMINATION</ideal_output>
<sample_input>Invoices are due net 45.</sample_input>
<ideal_output>PAYMENT</ideal_output>
<sample_input>We may suspend service for breach, and fees still accrue.</sample_input>
<ideal_output>TERMINATION</ideal_output>"""

# Same test, better prompt; wrap each clause in <clause> tags
good_rate, good_outs = valid_rate(SYSTEM, lambda c: f"<clause>{c}</clause>")
show("✅ Constrained prompt", f"Valid outputs: {good_rate:.0%}\n\nSamples: {sorted(set(good_outs))}", "green")

### Structured outputs: let the API enforce the format

A prompt is only a request, and Claude can still ignore it. A JSON schema is different because the API enforces it. Tokens that would break the schema simply can't be generated.

- `output_config.format` controls the final answer.
- `strict: true` on a tool controls the tool's arguments (we'll use this in Chapter 4).

There are a few trade-offs. The first call with a new schema is slower, because the API has to prepare it (it's cached for 24 hours after that). You pay a few extra input tokens. You still need to check `stop_reason`, because a refusal or hitting `max_tokens` can leave you with output that doesn't match. And it doesn't work together with message prefilling.

#### What are we going to do?

1. Define a schema. `category` has to be one of our four labels, and `summary` is a one-line summary.
2. Ask Claude to classify and summarize the tricky "suspend your access" clause, with the schema attached.
3. Check `stop_reason` before parsing. Try to make this a habit.
4. Parse the JSON and print it.

In [ ]:
# A form with rules: category must be one of our labels, summary is free text
schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": sorted(LABELS)},
        "summary": {"type": "string"},
    },
    "required": ["category", "summary"],
    "additionalProperties": False,
}

# Ask Claude to fill in the form (output_config enforces the schema)
r = client.messages.create(
    model=SONNET, max_tokens=300,
    output_config={"format": {"type": "json_schema", "schema": schema}},
    messages=[{"role": "user", "content": f"Classify and summarize in one sentence: <clause>{clauses[4]}</clause>"}],
)

# Stop if Claude didn't finish normally (e.g. was cut off)
assert r.stop_reason == "end_turn", f"Don't parse: stop_reason={r.stop_reason}"  # always check
# Convert the JSON text into a Python dict and display it
show("🧱 Schema-enforced output", json.loads(text_of(r)), "green")

> **Rule of thumb:** work out what went wrong, add the one technique that fixes it, then run it again. If three re-prompts in a row didn't help, stop adding text. Also keep it simple. A prompt like "summarize this paragraph" doesn't need any of this.

## Chapter 3 — Extended Thinking

Prompting changes *what* Claude writes. Thinking changes *how much work Claude does before it answers*. On current models, thinking is adaptive. Claude decides how much to think, and you guide it with `effort`. (The older `budget_tokens` setting is deprecated and gives an error on the newest models.)

| Task | Thinking? |
|---|---|
| Multi-step reasoning, planning dependent actions | ✅ On, with effort matched to difficulty |
| Classification, extraction, format conversion | ❌ Off. A constrained prompt is cheaper and just as accurate |
| Agent loops with tools | ✅ On, but follow the **carry-back rule** below |

Thinking tokens are billed like output tokens, so let's look at the actual cost.

#### What are we going to do?

1. Write one hard question: list every way the contract can end or access can be suspended, who can trigger each one, and which is worst for the customer.
2. Ask it twice on Sonnet. Once with thinking off, and once with adaptive thinking at high effort.
3. Print the block types, the number of output tokens and the start of each answer, so you can compare quality against cost.

In [ ]:
# One hard question, with the whole contract attached
hard_q = (
    "Using the contract below, list every way the contract can end or access can be suspended, "
    "who can trigger each, and which one gives the customer the least protection. Cite sections.\n\n" + contract_text
)

# Two setting sets to compare: thinking off vs adaptive thinking
runs = {
    "thinking OFF": dict(thinking={"type": "disabled"}),
    "adaptive · effort=high": dict(
        thinking={"type": "adaptive", "display": "summarized"}, output_config={"effort": "high"}
    ),
}

# Ask the same question twice with different settings (**kwargs unpacks them into the call)
for name, kwargs in runs.items():
    r = client.messages.create(model=SONNET, max_tokens=6000,
                               messages=[{"role": "user", "content": hard_q}], **kwargs)
    # Show block types, tokens used (= cost) and the start of each answer
    show(f"{name} · blocks={[b.type for b in r.content]} · output_tokens={r.usage.output_tokens}",
         text_of(r)[:700] + " …", "yellow" if "OFF" in name else "purple")

### The carry-back rule

When thinking is on and you also use tools, you have to send every `thinking` block back to the API exactly as you got it on the next turn. Each block has a signature, so if you edit it, shorten it or leave it out, the request is rejected. The same goes for redacted thinking blocks.

> Worried that the reasoning will fill up your context? Don't remove the blocks. Manage the context instead. We'll do that in Chapter 6.

In our loops we just append `response.content` as it is, which follows the rule without any extra work.

## Chapter 4 — Tool Use & Schema Design

This surprises a lot of people: Claude never runs your tools. It reads your schemas, decides which one fits, and sends back a `tool_use` block. Your code runs the tool and sends the result back as a `tool_result`.

```text
define schema → send message → tool_use block → YOU execute → tool_result → Claude continues
```

| Block | Rule |
|---|---|
| `text` | Keep it when you append the assistant turn |
| `tool_use` | Has a unique `id` |
| `tool_result` | Must be in the **very next user turn**, with the same `tool_use_id` (set `is_error: true` on failure) |
| `thinking` | Pass it back unchanged |

A schema has a `name`, a `description` and an `input_schema`. Claude picks a tool based on its description, so say when to use the tool and also when not to.

#### What are we going to do?

We'll give Claude two tools for working with our contract:

1. `search_contract` finds a keyword and returns short snippets with page numbers.
2. `get_page` returns the text of one page (cut short so it doesn't use up our context).

We'll put them in a lookup table called `IMPL`, so our code can run whichever one Claude asks for. Then we'll write two versions of the tool list. `tools_good` has clear descriptions. `tools_bad` has the same tools with one vague description. The next few cells compare them.

In [ ]:
# Tool 1: find a keyword and return short snippets with page numbers
def search_contract(keyword):
    """Return up to 5 short snippets (with page numbers) that contain the keyword."""
    hits = []
    for n, text in enumerate(pages, 1):
        i = text.lower().find(keyword.lower())
        if i >= 0:
            hits.append(f"[page {n}] …{text[max(0, i - 120): i + 200]}…".replace("\n", " "))
    return "\n".join(hits[:5]) or "No matches."


# Tool 2: return one page's text (capped to protect context)
def get_page(page):
    """Return the text of one page (capped at 2,500 characters to protect context)."""
    if not 1 <= page <= len(pages):
        raise ValueError(f"page must be 1–{len(pages)}")
    return pages[page - 1][:2500]


# Lookup table: tool name -> the real function to run
IMPL = {"search_contract": search_contract, "get_page": get_page}

# The menu Claude reads: name, description and arguments for each tool
tools_good = [
    {
        "name": "search_contract",
        "description": (
            "Search the contract for a keyword and return short snippets with page numbers. "
            "Use this when you don't yet know where a topic appears. "
            "Do not use this if you already know the page number — use get_page instead."
        ),
        "input_schema": {"type": "object", "properties": {"keyword": {"type": "string"}}, "required": ["keyword"]},
    },
    {
        "name": "get_page",
        "description": (
            "Return the full text of one contract page. Use this only when you already know the page number "
            "(for example from a search result). Do not use this to look for a topic."
        ),
        "input_schema": {"type": "object", "properties": {"page": {"type": "integer"}}, "required": ["page"]},
    },
]

# Same tools with overlapping, vague descriptions — Claude can't tell them apart
tools_bad = [
    {**t, "description": "Use this to find information in the contract."} for t in tools_good
]

**Which tool does Claude pick first?** Let's test it. We'll ask the same questions with vague descriptions and with clear ones. Vague descriptions don't fail every time. They fail on the in-between cases, and it gets worse as you add more tools.

#### What are we going to do?

1. Write a small helper that sends a question and returns the name of the first tool Claude picks.
2. Ask two questions. One needs a search ("Where does the contract discuss termination?") and one names a page ("Show me everything on page 3.").
3. Run both against `tools_bad` and `tools_good`, then compare what Claude picked with what we expected.

In [ ]:
# Ask one question and return the name of the first tool Claude picks
def first_tool(tools, question):
    r = client.messages.create(model=HAIKU, max_tokens=300, tools=tools,
                               messages=[{"role": "user", "content": question}])
    return next((b.name for b in r.content if b.type == "tool_use"), "none")


# (question, tool we expect)
tests = [("Where does the contract discuss termination?", "search_contract"),
         ("Show me everything on page 3.", "get_page")]

# Compare vague vs specific descriptions
for label, tools in [("vague", tools_bad), ("specific", tools_good)]:
    picks = [(first_tool(tools, q), expected) for q, expected in tests]
    show(f"{label} descriptions", "\n".join(f"picked {p:16} expected {e}" for p, e in picks),
         "red" if label == "vague" else "green")

### The tool-use loop

So far Claude has only picked a tool. Now we'll close the loop. The idea is simple: keep running tools until `stop_reason` is no longer `"tool_use"`. Claude can ask for several tools at once in a single turn, and all of those results go back together in one user message.

#### What are we going to do?

1. Write `run_tools()`, a loop that calls Claude, keeps the full reply in the history, runs the tools it asks for, and sends all the results back in one turn.
2. Handle errors safely. If a tool fails, we send the error to Claude instead of crashing.
3. Add a `max_iters` limit so the loop can't run forever.
4. Ask a real question: what notice period applies to termination for convenience, and on which page? Each 🔧 line you see is a tool call.

In [ ]:
# The agent loop: call Claude -> run requested tools -> send results back -> repeat
def run_tools(question, tools=tools_good, max_iters=6):
    # The conversation so far is Claude's only memory
    messages = [{"role": "user", "content": question}]
    # Cap on rounds so it can't loop forever
    for _ in range(max_iters):
        r = client.messages.create(model=SONNET, max_tokens=1500, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": r.content})   # keep ALL blocks, unchanged
        # Not asking for tools = Claude is done
        if r.stop_reason != "tool_use":
            return text_of(r), messages
        # Run every tool Claude asked for
        results = []
        for b in r.content:
            if b.type != "tool_use":
                continue
            try:
                out, err = IMPL[b.name](**b.input), False
            except Exception as e:                                       # report failures to Claude
                out, err = f"Error: {e}", True
            print(f"🔧 {b.name}({b.input}){'  ❌' if err else ''}")
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": str(out), "is_error": err})
        messages.append({"role": "user", "content": results})        # ALL results, one user turn
    raise RuntimeError("max_iters reached")


# Try it on a real question
answer, history = run_tools("What notice period applies to termination for convenience, and on which page is it?")
show("Answer", answer, "green")

### What a broken pairing looks like

Let's break something on purpose. It's much nicer to see this error here than in production.

#### What are we going to do?

1. Take the history from the run above and find the assistant message that contains a `tool_use`.
2. Build a new conversation where the `tool_result` has the wrong id (`toolu_WRONG`).
3. Send it and catch the error.

The API will reject it. No change to the prompt can fix a structural mistake like this.

In [ ]:
# Find Claude's first tool request in the history
first_assistant = next(m for m in history if m["role"] == "assistant"
                       and any(b.type == "tool_use" for b in m["content"]))
# Fake conversation: the tool_result has the WRONG id
broken = [
    history[0],
    first_assistant,
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_WRONG", "content": "Page text…"}]},
]

# Send it; the API should reject it
try:
    client.messages.create(model=SONNET, max_tokens=300, tools=tools_good, messages=broken)
except anthropic.BadRequestError as e:
    show("🚫 Validation error", e.message, "red")

**Schema design cheat-sheet**

| Decision | Guidance |
|---|---|
| Required fields | Only if the call makes no sense without them. Otherwise Claude invents values |
| Optional fields | Give them sensible defaults. Claude may omit what it doesn't know |
| Description length | 3–4 sentences: what it does, when to use it, when *not* to, and what it returns |
| Dependent calls | Keep them in separate turns. Independent calls run in parallel (`disable_parallel_tool_use` forces one at a time) |
| Near-duplicate tools | Add exclusions, or merge them into one tool with a `type` parameter |
| `strict: true` | Validates the arguments against your schema before your code runs |

Coming up in Module 3: MCP lets you connect ready-made tool servers instead of writing schemas by hand. The loop stays the same, but every server you connect adds more tool definitions to your context.

## Chapter 5 — Streaming Without Corrupting State

When you stream, Claude doesn't send one finished message. It sends a series of events, and you put the message together from them.

| Event | Your handler |
|---|---|
| `message_start` | Create an empty content list |
| `content_block_start` | Open a slot (a `tool_use` block opens with its name and id, but **no input yet**) |
| `content_block_delta` | Append the fragment (tool input arrives as **partial JSON**) |
| `content_block_stop` | The block is complete. This is the **first moment** tool JSON can be parsed |
| `message_delta` | Record the `stop_reason` |
| `message_stop` | The whole message is done. **Only now** do you commit it to history |

> A stream ending is not the same as a message finishing. If the connection drops in the middle of a `tool_use`, a half-built block ends up in your history and breaks the next request. The error will point at the wrong place, which makes it hard to find.

#### What are we going to do?

We'll write `stream_turn()`, a handler that follows those rules. It will:

1. Build each block piece by piece as the events arrive, and print the text live.
2. Wait for `content_block_stop` before it parses any tool JSON.
3. Return the content only if it saw `message_stop`. Otherwise it says `interrupted`.
4. Have a `drop_after` option so we can fake a dropped connection.

We'll start with the normal case, a stream that finishes cleanly.

In [ ]:
def stream_turn(messages, tools, drop_after=None):
    """Stream one turn. Returns (content, stop_reason), or (None, 'interrupted') if the stream never reached message_stop."""
    # blocks = scratch pad where we rebuild the message from stream events
    blocks, stop_reason, finished, seen = {}, None, False, 0
    with client.messages.create(model=SONNET, max_tokens=1000, tools=tools, messages=messages, stream=True) as stream:
        for ev in stream:
            seen += 1
            # A new piece (text or tool call) starts: open an empty slot
            if ev.type == "content_block_start":
                b = ev.content_block
                blocks[ev.index] = {"type": b.type, "id": getattr(b, "id", None), "name": getattr(b, "name", None), "text": "", "json": ""}
            # A fragment arrived: text is printed live, tool input is partial JSON (not valid yet)
            elif ev.type == "content_block_delta":
                d, blk = ev.delta, blocks[ev.index]
                if d.type == "text_delta":
                    blk["text"] += d.text
                    print(d.text, end="", flush=True)
                elif d.type == "input_json_delta":
                    blk["json"] += d.partial_json            # NOT valid JSON yet
            # Piece finished: only now is tool JSON safe to parse
            elif ev.type == "content_block_stop":
                blk = blocks[ev.index]
                if blk["type"] == "tool_use":
                    blk["input"] = json.loads(blk["json"] or "{}")   # safe only now
            elif ev.type == "message_delta":
                stop_reason = ev.delta.stop_reason
            # Whole message received
            elif ev.type == "message_stop":
                finished = True
            if drop_after and seen >= drop_after:               # simulate a dropped connection
                break
    # Never saw message_stop = the stream was cut off, so return nothing
    if not finished:
        return None, "interrupted"
    # Stream completed: assemble clean blocks to put in history
    content = []
    for i in sorted(blocks):
        b = blocks[i]
        content.append({"type": "text", "text": b["text"]} if b["type"] == "text"
                       else {"type": "tool_use", "id": b["id"], "name": b["name"], "input": b["input"]})
    return content, stop_reason


# Test conversation
q = [{"role": "user", "content": "Find the termination for convenience clause, then explain it in 2 sentences."}]

# Run a normal stream
print("— healthy stream —")
content, reason = stream_turn(q, tools_good)
print(f"\n\n→ stop_reason={reason}, blocks={[b['type'] for b in content]}")

Now for the real test. What happens when the connection drops?

#### What are we going to do?

1. Run `stream_turn()` again, but with `drop_after=8`, so the stream gets cut off after only 8 events.
2. Check whether we got a complete turn back. We didn't, so we throw the partial turn away.
3. Confirm that our history hasn't changed, which means it's safe to try again.

This is the "only commit after `message_stop`" rule in action.

In [ ]:
print("— stream dropped mid-way —")
# Copy the conversation so we can prove it stays untouched
history2 = list(q)
# Stop after 8 events to simulate a dropped connection
content, reason = stream_turn(history2, tools_good, drop_after=8)

# Incomplete turn: throw it away, nothing added to history
if content is None:                       # gate on message_stop
    print(f"\n\n→ {reason}: partial turn DISCARDED, history untouched ({len(history2)} message). Safe to retry.")
else:
    history2.append({"role": "assistant", "content": content})

**Rules for your stream handler:**

1. Only use a block after you've seen its `content_block_stop`.
2. Only add to the history after `message_stop`.
3. If the stream is interrupted, throw away the partial turn and retry from the last complete one.
4. Check `stop_reason == "tool_use"` before you keep looping.

If your responses are short and only used by your backend, a normal non-streamed call is simpler and avoids all of this.

## Chapter 6 — Context Engineering

Think of the context window as a fixed budget. Everything has to fit in it: the system prompt, the history, the tool definitions, every tool result and the answer. Tool results use up the most. In production they are often 3 to 5 times longer than the ones you tested with, so a session that lasts 50 turns in development can run out of room at turn 8. And it won't look like running out of room. It will look like Claude picking the wrong tool, which is hard to spot.

**About choosing a model:** start with Sonnet. Move up to Opus only if your tests show Sonnet isn't good enough. Move down to Haiku only if your tests show the drop in quality is fine for you.

Let's measure this instead of guessing.

#### What are we going to do?

1. Write `n_tokens()`, which asks the API to count the tokens in a text before we send it.
2. Set a small budget of 6,000 tokens so we can see the effect quickly.
3. Pretend an agent reads the contract one page per turn, and add up the tokens as it goes.
4. Print a growing bar for each turn and stop with a warning as soon as we go over the budget.

In [ ]:
# Ask the API how many tokens a text is (no answer generated)
def n_tokens(text):
    return client.messages.count_tokens(model=SONNET, messages=[{"role": "user", "content": text}]).input_tokens


BUDGET = 6_000          # a deliberately small team-imposed budget, to see the effect
# Simulate an agent reading one page per turn and watch the token total grow
used = n_tokens(SYSTEM)
for turn, page in enumerate(range(1, len(pages) + 1), 1):
    used += n_tokens(get_page(page))
    print(f"turn {turn:2}: +page {page:2} → {used:6,} tokens {'█' * (used // 400)}")
    # Over budget: warn and stop
    if used > BUDGET:
        print(f"\n🚨 Budget of {BUDGET:,} exceeded at turn {turn}. Time to prune/compact — before the symptoms start.")
        break

### Four ways to stay in budget

| Strategy | What it does | Use when | You lose |
|---|---|---|---|
| **Pruning** | Rewinds or drops spent tool results | Dead-end path, finished lookups | Work after the rewind point |
| **Compaction** (`/compact`, server-side, or your own summarizer) | Summarizes the history | You're near the ceiling and the same task continues | Anything the summary leaves out |
| **Clearing** (`/clear`, new session) | Empties the context | The next task is unrelated | Everything you didn't persist |
| **Subagent handoff** | Uses an isolated context and returns a summary | Self-contained or exploratory subtasks | Visibility into its steps |

Compaction is only as good as the prompt that writes the summary. Let's see for ourselves.

#### What are we going to do?

1. Take a short transcript with a few traps in it: a file edited by mistake, an error, and a revert.
2. Summarize it twice with Haiku. Once with a generic prompt, and once with a specific prompt that says what to keep (file paths, decisions, and errors with how they were fixed).
3. Compare the two summaries. Does the generic one still say that `staging.yaml` was reverted?

In [ ]:
# A short fake conversation with traps (a mistaken edit, an error, a revert)
transcript = """
user: Please update the rate limit in config/prod.yaml from 100 to 500.
assistant: I edited config/prod.yaml (rate_limit: 500) and config/staging.yaml (rate_limit: 500).
user: validate_config failed: 'burst' must be <= rate_limit.
assistant: Fixed by setting burst: 400 in config/prod.yaml. validate_config now passes.
user: Don't touch staging, that was a mistake. Revert it.
assistant: Reverted config/staging.yaml to rate_limit: 100.
"""

# Two summarizer prompts: vague vs. says exactly what to keep
generic = "Summarize the conversation so far.\n" + transcript
specific = ("Summarize the conversation, preserving: all file paths modified, all decisions made, "
            "and any errors encountered and how they were resolved.\n" + transcript)

# Summarize with each and compare
for name, p in [("generic summarizer", generic), ("specific summarizer", specific)]:
    r = client.messages.create(model=HAIKU, max_tokens=300, messages=[{"role": "user", "content": p}])
    show(name, text_of(r), "red" if name.startswith("generic") else "green")

### Prompt caching: pay once for the part that doesn't change

If you send the same big chunk of text with every request, you pay for it every time. Prompt caching fixes that. Mark the last block you want cached with `cache_control: {"type": "ephemeral"}`. You can have up to 4 of these markers, and the order is tools, then system, then messages. A few rules:

- The cached part has to match exactly.
- The default lifetime is 5 minutes (you can opt in to `"ttl": "1h"`).
- There's a minimum size, about 1,024 tokens on most models. Shorter prompts just won't be cached, and you won't get an error.

#### What are we going to do?

1. Put the whole contract in the system prompt and mark it to be cached.
2. Ask two different questions about it, one after the other.
3. Print the cache numbers for each request: how many tokens were written to the cache, how many were read from it, and how many were billed as normal input.

In [ ]:
# Put the whole contract in the system prompt and mark it cacheable
system_blocks = [
    {"type": "text", "text": "You answer questions about the contract below.\n\n" + contract_text,
     "cache_control": {"type": "ephemeral"}}
]

# Ask two questions that share the same cached prefix
for i, q in enumerate(["Who can terminate the agreement?", "What happens to customer content after termination?"], 1):
    r = client.messages.create(model=SONNET, max_tokens=200, system=system_blocks,
                               messages=[{"role": "user", "content": q}])
    # Cache tokens: written (1st request), read (2nd request), or billed normally
    u = r.usage
    print(f"request {i}: cache_write={u.cache_creation_input_tokens or 0:>6,}  cache_read={u.cache_read_input_tokens or 0:>6,}  uncached_input={u.input_tokens:>5,}")

Look at the numbers. Request 1 writes to the cache, and request 2 reads from it at a much lower price. If your sessions have several turns with the same system prompt and tool list, caching is one of the easiest ways to cut cost. And `count_tokens`, which we used above, lets you catch requests that are too big before they fail.

One last thing. Subagents start from scratch. They only get the task and the context you give them, and they don't inherit your skills.

## Chapter 7 — Building a Production Agent

An agent is a tool-use loop that runs for several steps, with managed context and a goal. That power comes with more complexity and more ways to fail, so first decide if you really need one:

| Choose a **workflow** when… | Choose an **agent** when… |
|---|---|
| You can list the exact steps in code | You can specify the goal and the tools, but not the path |
| Error cost is real and step guardrails matter | Inputs vary unpredictably |
| Every run follows the same sequence | The task needs creative sequencing of tools |

> Progression: single API call → workflow → agent. Only move up when the simpler pattern can't cope.

**Who runs the loop?** It's the same pattern, wired up in three different ways:

| Path | You own | Pick when |
|---|---|---|
| **Raw Messages API loop** | Everything | You want full control, or you're learning the loop |
| **Agent SDK** | Tool execution + your app (the SDK runs the loop in your process) | You want Claude Code's loop; set `settingSources` explicitly |
| **Managed Agents** (beta) | Your app + the agent definition (Anthropic runs the loop and sandbox) | Long-running work. ⚠️ Server-side state means it's not eligible for ZDR / HIPAA BAA |

**Loop checklist:** ① tools registered ② system prompt scoped to the task ③ every `tool_use` answered ④ **a human gate before irreversible actions** ⑤ explicit exit conditions.

**Where does a human check go?** Ask yourself: "What's the worst that can happen if this step runs without anyone checking?" Put a check before anything that deletes, writes or sends, after a plan, and whenever the output is unexpected.

#### What are we going to do?

We'll build a small contract-review agent that has one risky tool. `send_report` "emails" a result, and you can't take an email back, so it needs approval first.

1. Add `send_report` to our tools. Its description says it can't be undone and when not to use it. (Here it only saves the email to a list, so nothing is really sent.)
2. Mark it as `NEEDS_APPROVAL` and write `human_approves()`, a checkpoint that pauses and asks you y/N.
3. Write `run_agent()` with two ways to stop: a limit on the number of rounds, or Claude finishing on its own.
4. Give it a goal: find the termination notice rules and email a 3-bullet summary. When the checkpoint box shows up, type your answer.

Run it twice. Answer **y** once and **N** once.

In [ ]:
# Fake outbox: stands in for a real, irreversible email
sent_reports = []


def send_report(to, body):
    sent_reports.append((to, body))          # stand-in for a real, irreversible email
    return f"Report sent to {to}."


# Register the new tool so our loop can run it
IMPL["send_report"] = send_report

# Tool menu = read tools + the dangerous send_report
agent_tools = tools_good + [{
    "name": "send_report",
    "description": (
        "Email a final report to a recipient. This is irreversible. Use it once, only after the analysis "
        "is complete. Do not use it for drafts or intermediate notes."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"to": {"type": "string"}, "body": {"type": "string"}},
        "required": ["to", "body"],
    },
}]

# Tools that must be approved by a human first
NEEDS_APPROVAL = {"send_report"}


# Show what the agent wants to do and ask you y/N
def human_approves(name, args):
    show("✋ Human checkpoint", f"Agent wants to run {name}\n\n{json.dumps(args, indent=2)[:600]}", "yellow")
    return input("Approve? (y/N): ").strip().lower() == "y"


# Same loop as before, plus a human gate and two exit conditions
def run_agent(goal, max_iters=10):
    messages = [{"role": "user", "content": goal}]
    system = ("You are a contract-review agent. Tools: search_contract, get_page, send_report. "
              "Research with the read tools, then send exactly one report. Stop when the report is sent.")
    for i in range(max_iters):                                     # exit condition #1: iteration cap
        r = client.messages.create(model=SONNET, max_tokens=1500, system=system, tools=agent_tools, messages=messages)
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":                            # exit condition #2: model is done
            return text_of(r), messages
        results = []
        for b in r.content:
            if b.type != "tool_use":
                continue
            # Gate: ask the human; if refused, tell Claude instead of running the tool
            if b.name in NEEDS_APPROVAL and not human_approves(b.name, b.input):
                results.append({"type": "tool_result", "tool_use_id": b.id, "is_error": True,
                                "content": "Rejected by human reviewer. Do not retry; ask what to change."})
                continue
            try:
                out, err = str(IMPL[b.name](**b.input)), False
            except Exception as e:
                out, err = f"Error: {e}", True
            print(f"🔧 [{i}] {b.name}({str(b.input)[:80]})")
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": out, "is_error": err})
        messages.append({"role": "user", "content": results})
    raise RuntimeError("Agent hit max_iters without finishing")


# Run it - type y once and N once to see both paths
final, _ = run_agent("Find the termination notice rules and email a 3-bullet summary to legal@example.com.")
show("Agent result", final, "green")
print("Reports actually sent:", len(sent_reports))

Did you try both answers? When you said N, the refusal went back to Claude as an error result and the program didn't crash. Claude can read it, adjust, and ask what to change. That's what a good approval step looks like.

> **Too many tools** is a common mistake. If you add every tool "just in case", Claude gets worse at picking the right one. Start with the fewest tools you can, and add one only when you know something is missing.

**If you handle regulated data, that decides your setup before anything else.** HIPAA needs a setup covered by a BAA (not Console, Workbench or consumer plans). EU data residency means Bedrock or Vertex with a fixed region. FedRAMP means Claude for Government, Bedrock GovCloud or Vertex Assured Workloads. Privileged data means the direct API through the firm's own logged gateway. (Module 4 goes deeper into security.)

## Chapter 8 — Agent Memory

How much your agent should remember depends on how your sessions actually look, not on what's easiest to code:

| Scope | Persists | Cost | Use when |
|---|---|---|---|
| **In-context** | Within one session | Tokens grow every turn | Short, one-off sessions |
| **External storage** | Across sessions and users | Retrieval latency + code | Multi-day threads, shared state |
| **Summarized** | Condensed history | Cheap, but lossy | Long conversations |
| **Stateless** | Nothing | None | Self-contained jobs |

A common mistake looks like this. In development you used one long session, but in production there are lots of short ones, and by session 4 the history is already 40k tokens. Fixing that later takes hours. Choosing the right approach at the start takes minutes.

#### What are we going to do?

We'll build a small version of external storage with a summarizer that keeps only what matters:

1. Write `save_memory()` and `load_memory()`, which keep one summary per case in a small JSON file.
2. **Session 1:** take a short conversation about case C-17 and ask Haiku to summarize it. We tell it exactly what to keep: the case id, the concern, the decisions and the open questions. Then we save the summary.
3. **Session 2:** start fresh with no history at all. Load the saved summary into the system prompt and ask Claude what we concluded.

If Claude remembers Session 1 correctly, the memory works.

In [ ]:
# Memory lives in a small JSON file: one summary per case id
MEMORY_FILE = "/content/agent_memory.json"


def save_memory(case_id, summary):
    mem = json.load(open(MEMORY_FILE)) if os.path.exists(MEMORY_FILE) else {}
    mem[case_id] = summary
    json.dump(mem, open(MEMORY_FILE, "w"), indent=2)


def load_memory(case_id):
    return json.load(open(MEMORY_FILE)).get(case_id, "") if os.path.exists(MEMORY_FILE) else ""


# ── Session 1 (illustrative transcript) ──
s1 = [
    {"role": "user", "content": "Review the AWS termination terms for case C-17. Key concern: data deletion."},
    {"role": "assistant", "content": "After termination the customer has 30 days to retrieve content. Open question: what happens to backups?"},
]
# Ask Haiku to keep only what the next session needs
summary = text_of(client.messages.create(model=HAIKU, max_tokens=250, messages=[{"role": "user", "content":
    "Summarize for the next session, preserving: the case id, the user's concern, decisions made, open questions.\n\n"
    + json.dumps(s1)}]))
# Store it on disk
save_memory("C-17", summary)

# ── Session 2 (fresh process, no history) ──
# Fresh start: no history, only the saved summary in the system prompt
r = client.messages.create(model=HAIKU, max_tokens=200,
    system="Prior context for this case:\n" + load_memory("C-17"),
    messages=[{"role": "user", "content": "Remind me what we concluded and what is still open."}])
show("Session 2 — recalls Session 1 from storage", text_of(r), "green")

### Skills: reusable instructions that load *on demand*

A Skill is a `SKILL.md` file. It has a short header with a `name` and `description`, followed by the steps. Claude only reads the names and descriptions at first, and loads the full file when a request matches. So a skill costs almost nothing until you actually need it.

| Pattern | Loads | Context cost | Best for |
|---|---|---|---|
| **Skill** | On description match | Low | Task-specific procedures |
| **CLAUDE.md** | Every session | Fixed, always | Always-on project rules |
| **In-context** | Every turn | Grows with the session | One-off, short work |

Skills on the Messages API are in beta (headers `code-execution-2025-08-25` and `skills-2025-10-02`, running in a code-execution container). Subagents don't inherit skills, so list them explicitly. Check the headers against the current docs before you build on them.

## Chapter 9 — Multimodal & Batch Ingestion

This cost catches a lot of people off guard: images use tokens before Claude reads a single word. Each 28×28-pixel patch is about 1 token, so an image costs roughly `⌈w/28⌉ × ⌈h/28⌉` tokens. Every model also has a maximum resolution, and bigger images get shrunk first. Work out the cost using your biggest real image, not your small test ones. Adding a resize step takes 10 minutes before launch and much longer after.

#### What are we going to do?

1. Write a helper that makes a blank white image of any size and wraps it as an image block. We only care about the size here, not what's in it.
2. Work out the token estimate from the formula for three sizes: 280×280, 1000×1000 and 1400×1400.
3. Ask the API what it really counts, and compare the two numbers. If they differ, that's the resizing and limits at work.

In [ ]:
from PIL import Image
import io


# Make a blank image of a given size and wrap it the way the API expects (base64)
def image_block(w, h):
    buf = io.BytesIO()
    Image.new("RGB", (w, h), "white").save(buf, "PNG")
    return {"type": "image", "source": {"type": "base64", "media_type": "image/png",
            "data": base64.standard_b64encode(buf.getvalue()).decode()}}


# Token cost of the text alone, so we can subtract it
baseline = n_tokens("Describe this image.")
# Compare our formula with what the API really counts
for w, h in [(280, 280), (1000, 1000), (1400, 1400)]:
    # Formula: one token per 28x28-pixel patch
    est = math.ceil(w / 28) * math.ceil(h / 28)
    real = client.messages.count_tokens(model=SONNET, messages=[{"role": "user", "content": [
        image_block(w, h), {"type": "text", "text": "Describe this image."}]}]).input_tokens - baseline
    print(f"{w}×{h}: formula ≈ {est:5,} tokens | API reports ≈ {real:5,} (limits/downscaling may differ)")

### How to send an asset

| Method | Overhead | Use when |
|---|---|---|
| **Inline base64** | Full payload on every request | A one-off image |
| **URL** | None, but it must be public, stable and reachable | It's already hosted at a stable URL |
| **Files API** (beta; not on Bedrock/Vertex) | Upload once, then send just a `file_id` | You reuse the same asset, or manage assets separately from inference |

PDFs use a `document` block (same three source types, with an optional `title` and `context`). Everything you learned about prompting still applies. Images can be ambiguous, so tell Claude what to do ("if objects overlap, describe each one and note the overlap").

### Batches: for big jobs that can wait

Looping over the normal API is not batching. It's just one call after another, and you'll still hit rate limits. The Message Batches API takes up to 100,000 requests (or 256 MB), runs in the background (within 24 hours), costs less per token, and returns results in any order, so you match them up using `custom_id`.

| Use case | API |
|---|---|
| A user is waiting for a classification | Sync |
| A nightly job on 5,000 records | **Batch** |
| Evaluating a new prompt on 2,000 examples | **Batch** |
| A chat reply | Sync |

#### What are we going to do?

We'll take the same five clauses from Chapter 2 and classify them as a batch:

1. Send all five as one batch job. Each one gets a `custom_id` (`clause-0`, `clause-1`, …).
2. Wait and check the status every 10 seconds until it's done. This can take a few minutes, so be patient.
3. Put the results in a dictionary using `custom_id` as the key, then print each label next to its clause. We match by id and not by position, because the order isn't guaranteed.

In [ ]:
# Send all clauses as one background job; custom_id tells the answers apart
batch = client.messages.batches.create(requests=[
    {"custom_id": f"clause-{i}",
     "params": {"model": HAIKU, "max_tokens": 20, "system": SYSTEM,
                "messages": [{"role": "user", "content": f"<clause>{c}</clause>"}]}}
    for i, c in enumerate(clauses)
])
print("Submitted batch", batch.id, "— this can take a few minutes…")

# Check every 10 seconds until the batch has ended
while (status := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    time.sleep(10)

# Keep successful answers, keyed by custom_id
results = {item.custom_id: text_of(item.result.message).strip()
           for item in client.messages.batches.results(batch.id) if item.result.type == "succeeded"}
for i, c in enumerate(clauses):                       # re-join by custom_id, NOT by position
    print(f"{results.get(f'clause-{i}', '?'):12} ← {c[:60]}…")

## 🏁 Cheat Sheet

That's the whole lab. Here's a one-table summary you can come back to.

| Chapter | One-line takeaway |
|---|---|
| Prompting | Diagnose the failure type, add the one missing technique, and use structured outputs when the parser must never break |
| Thinking | On for hard multi-step work, off for lookups; return thinking blocks unchanged |
| Tools | Descriptions drive selection, so say when *not* to use a tool; every `tool_use` needs a matching `tool_result` in the next user turn |
| Streaming | Stream end ≠ message complete; commit only after `message_stop` |
| Context | Measure real tool outputs; prune, compact (with a good summarizer), cache stable prefixes, hand off to subagents |
| Agents | Workflow vs agent first; gate irreversible actions with a human; set explicit exit conditions |
| Memory | Choose the scope at design time; Skills load on demand |
| Multimodal/Batch | Price images before building; batch ≠ looping; join by `custom_id` |

**Next:** *Claude Code, MCP & Integration*, where we make this integration configurable, shareable and safe.